In [2]:
import pandas as pd
import numpy as np
import json

In [3]:
import json
with open("../citation.json", 'r') as f:
    citation = json.load(f)
citation

{'doi': 'http://doi.org/10.1681/ASN.2018020125',
 'citation': 'Wu, H., Malone, A.F., Donnelly, E.L., Kirita, Y., Uchimura, K., Ramakrishnan, S.M., Gaut, J.P. and Humphreys, B.D., 2018. Single-cell transcriptomics of a human kidney allograft biopsy specimen defines a diverse inflammatory response. Journal of the American Society of Nephrology, 29(8), pp.2069-2080.',
 'tables': ['https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6065085/bin/ASN.2018020125SupplementaryData3.xls']}

## Define the metadata

In [4]:
organism = "homo_sapiens"
cell_source = "kidney"
cell_state = None

reference = "GRCh38"

table_name = "clean_degs.xlsx" # local name

## Read in file

In [5]:
excel = pd.read_excel(table_name, sheet_name=None)
ct = {i: i.split('. ')[-1] for i in excel.keys()}

# stacks the sheets together and makes a new column "cell_type" from the sheet name
df = pd.concat(
    excel, keys=list(excel.keys())
    ).reset_index(0).rename(
        columns={"level_0": "celltype_id"}
        )
# # rename the cell types to be human readable
df["celltype"] = df["celltype_id"].map(ct)


In [8]:
df.head()

,celltype_id,gene,p_val,avg_logFC,pct.1,pct.2,p_val_adj,celltype
0,1. PT,GPX3,6.780000e-106,1.958635,0.442,0.109,1.390000e-101,PT
1,1. PT,CUBN,5.490000e-136,1.821094,0.417,0.046,1.120000e-131,PT
2,1. PT,CDH6,9.110000e-154,1.764531,0.464,0.035,1.870000e-149,PT
3,1. PT,LRP2,1.530000e-144,1.736607,0.480,0.049,3.140000e-140,PT
4,1. PT,PDZK1IP1,1.010000e-133,1.673290,0.431,0.038,2.060000e-129,PT


## Unfiltered

In [9]:
unfiltered_df = df.copy()
unfiltered_df.rename(columns ={"celltype": "cell_type_label", "gene": "feature_name"}, inplace=True)
unfiltered_df["organism"] = organism
unfiltered_df["cell_source"] = cell_source
unfiltered_df["cell_state"] = cell_state
unfiltered_df["feature_identifier"] = None

unfiltered_df.drop(['p_val', 'avg_logFC', 'pct.1', 'pct.2', 'p_val_adj'], axis=1, inplace=True) 
result = [] 

for _, row in unfiltered_df.iterrows():
    result.append({"extracted": {"organism": row["organism"], "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene"},
                   "derived": {"organism": row["organism"], "cell_type_id": None, "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene", "feature_identifier": row["feature_identifier"], "feature_type": "ensembl"},
                   "source": {"source_type": "deg", "source_rationale": "unfiltered", "source_id": "deg"}})

result

[{'extracted': {'organism': 'homo_sapiens',
   'cell_type_label': 'PT',
   'cell_source': 'kidney',
   'cell_state': None,
   'feature_name': 'GPX3',
   'feature_type': 'gene'},
  'derived': {'organism': 'homo_sapiens',
   'cell_type_id': None,
   'cell_type_label': 'PT',
   'cell_source': 'kidney',
   'cell_state': None,
   'feature_name': 'GPX3',
   'feature_type': 'ensembl',
   'feature_identifier': None},
  'source': {'source_type': 'deg',
   'source_rationale': 'unfiltered',
   'source_id': 'deg'}},
 {'extracted': {'organism': 'homo_sapiens',
   'cell_type_label': 'PT',
   'cell_source': 'kidney',
   'cell_state': None,
   'feature_name': 'CUBN',
   'feature_type': 'gene'},
  'derived': {'organism': 'homo_sapiens',
   'cell_type_id': None,
   'cell_type_label': 'PT',
   'cell_source': 'kidney',
   'cell_state': None,
   'feature_name': 'CUBN',
   'feature_type': 'ensembl',
   'feature_identifier': None},
  'source': {'source_type': 'deg',
   'source_rationale': 'unfiltered',
   's

## Perform the filtering

In [12]:
col_pval = "p_val"
col_lfc = "avg_logFC"
col_gene = "gene"
col_ct = "celltype"
col_rank = "pct.1"

In [13]:
min_mean = 100
max_pval = 1e-10
min_lfc = 1
max_gene_shares = 4
max_per_celltype = 20

# filter by pval and lfc
dfc = df.query(f"{col_pval} <= {max_pval} & {col_lfc} >= {min_lfc}")

# mask out genes that are shared between max_gene_shares cell types
non_repeat_genes = dfc[col_gene].value_counts()[dfc[col_gene].value_counts() < max_gene_shares].index.values

if col_rank:
    m = dfc[dfc[col_gene].isin(non_repeat_genes)].sort_values(col_rank, ascending = True)
else:
    m = dfc[dfc[col_gene].isin(non_repeat_genes)]

# max number to sample is equal to the min number of genes across all celltype
n_sample = min(m[col_ct].value_counts().max(), max_per_celltype)

# sample n_sample genes
markers = m.groupby(col_ct).tail(n_sample)
markers_dict = markers.groupby(col_ct)[col_gene].apply(lambda x: list(x)).to_dict()

## Check how many genes per cell type

In [14]:
markers[col_ct].value_counts()

celltype
Plasma2          20
LOH (DL)         20
PT               20
LOH (AL)         20
Pericyte         20
EC               20
B cells          20
CD               20
Myofibroblast    20
Mono2            20
Fibroblast       20
Mono1            20
Mast cells       20
Cycling cells    19
Plasma1          15
T cells          14
Name: count, dtype: int64

In [15]:
if col_rank:
    print(markers.groupby(col_ct)[col_rank].mean().sort_values())

celltype
T cells          0.327500
Cycling cells    0.439368
PT               0.440950
LOH (DL)         0.463800
EC               0.509750
Pericyte         0.518850
LOH (AL)         0.576600
Plasma1          0.604867
B cells          0.643500
CD               0.645600
Mono2            0.657950
Plasma2          0.657950
Myofibroblast    0.688750
Fibroblast       0.703450
Mono1            0.819900
Mast cells       0.918000
Name: pct.1, dtype: float64


In [16]:
markers[col_ct].value_counts()

celltype
Plasma2          20
LOH (DL)         20
PT               20
LOH (AL)         20
Pericyte         20
EC               20
B cells          20
CD               20
Myofibroblast    20
Mono2            20
Fibroblast       20
Mono1            20
Mast cells       20
Cycling cells    19
Plasma1          15
T cells          14
Name: count, dtype: int64

## Find Ensembl ID

In [17]:
import sys
import time

from urllib.parse import urlparse, urlencode
from urllib.request import urlopen, Request
from urllib.error import HTTPError

In [18]:
class EnsemblRestClient(object):
    def __init__(self, server='http://rest.ensembl.org', reqs_per_sec=5):
        self.server = server
        self.reqs_per_sec = reqs_per_sec
        self.req_count = 0
        self.last_req = 0

    def perform_rest_action(self, endpoint, hdrs=None, params=None):
        if hdrs is None:
            hdrs = {}

        if 'Content-Type' not in hdrs:
            hdrs['Content-Type'] = 'application/json'

        if params:
            endpoint += '?' + urlencode(params)

        data = None

        # check if we need to rate limit ourselves
        if self.req_count >= self.reqs_per_sec:
            delta = time.time() - self.last_req
            if delta < 1:
                time.sleep(1 - delta)
            self.last_req = time.time()
            self.req_count = 0
        
        try:
            request = Request(self.server + endpoint, headers=hdrs)
            response = urlopen(request)
            content = response.read()
            if content:
                data = json.loads(content)
            self.req_count += 1

        except HTTPError as e:
            # check if we are being rate limited by the server
            if e.code == 429:
                if 'Retry-After' in e.headers:
                    retry = e.headers['Retry-After']
                    time.sleep(float(retry))
                    self.perform_rest_action(endpoint, hdrs, params)
            else:
                sys.stderr.write('Request failed for {0}: Status code: {1.code} Reason: {1.reason}\n'.format(endpoint, e))
           
        return data

    def get_id(self, species, symbol):
        genes = self.perform_rest_action(
            endpoint='/xrefs/symbol/{0}/{1}'.format(species, symbol), 
            params={'object_type': 'gene'}
        )
        if genes:
            stable_id = genes[0]['id']
            return stable_id
        else:
            return "gene not found"

In [13]:
def run(symbol):
    client = EnsemblRestClient()
    gene_id = client.get_id('human', symbol)
    return gene_id

## Convert to evidence json

In [14]:
df = markers[[col_ct, col_gene]].rename(columns={col_ct : "cell_type_label", col_gene: "feature_name"})
df["organism"] = organism
df["cell_source"] = cell_source
df["cell_state"] = cell_state
df["feature_identifier"] = df["feature_name"]
df["feature_identifier"] = df["feature_identifier"].apply(run)
result = [] 

for _, row in df.iterrows():
    result.append({"extracted": {"organism": row["organism"], "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene"},
                   "derived": {"organism": row["organism"], "cell_type_id": None, "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene", "feature_identifier": row["feature_identifier"], "feature_type": "ensembl"},
                   "source": {"source_type": "deg", "source_rationale": "unfiltered", "source_id": "deg"}})

result

In [15]:
save

[{'cell_type_label': 'T cells',
  'gene': 'CD96',
  'organism': 'homo_sapiens',
  'cell_source': 'kidney',
  'cell_state': None,
  'gene_id': 'ENSG00000153283'},
 {'cell_type_label': 'Plasma2',
  'gene': 'IGLL1',
  'organism': 'homo_sapiens',
  'cell_source': 'kidney',
  'cell_state': None,
  'gene_id': 'ENSG00000128322'},
 {'cell_type_label': 'T cells',
  'gene': 'TOX',
  'organism': 'homo_sapiens',
  'cell_source': 'kidney',
  'cell_state': None,
  'gene_id': 'ENSG00000198846'},
 {'cell_type_label': 'Plasma1',
  'gene': 'IGKV4-1',
  'organism': 'homo_sapiens',
  'cell_source': 'kidney',
  'cell_state': None,
  'gene_id': 'ENSG00000211598'},
 {'cell_type_label': 'T cells',
  'gene': 'BCL11B',
  'organism': 'homo_sapiens',
  'cell_source': 'kidney',
  'cell_state': None,
  'gene_id': 'ENSG00000127152'},
 {'cell_type_label': 'Plasma2',
  'gene': 'IGLJ3',
  'organism': 'homo_sapiens',
  'cell_source': 'kidney',
  'cell_state': None,
  'gene_id': 'ENSG00000211678'},
 {'cell_type_label': '

## Save evidence

In [16]:
with open("evidence.json", "w") as f:
    json.dump(save, f, indent = 4) 